# Book Recommendation Experiment with DataVint

This notebook demonstrates:
1. Downloading real dataset from Kaggle using kagglehub
2. Data versioning with DataVint
3. Feature engineering (book_age, age_group)
4. XGBoost model training with hyperparameter sweeps
5. Lineage tracking in bipartite graph

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import datavint as dv
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score
import kagglehub
import os

## Step 1: Download Dataset from Kaggle

In [ ]:
# Download latest version
path = kagglehub.dataset_download("arashnic/book-recommendation-dataset")

print("Path to dataset files:", path)
print("\nFiles in dataset:")
for file in os.listdir(path):
    if file.endswith('.csv'):
        file_path = os.path.join(path, file)
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"  - {file} ({size_mb:.2f} MB)")

## Step 2: Load and Merge Data

In [ ]:
# Load ratings (full dataset)
df_ratings = pd.read_csv(
    os.path.join(path, "Ratings.csv")
)

print(f"Ratings shape: {df_ratings.shape}")
print(f"\nFirst 5 rows:")
df_ratings.head()

In [ ]:
# Load books
df_books = pd.read_csv(
    os.path.join(path, "Books.csv"),
    usecols=['ISBN', 'Year-Of-Publication', 'Book-Author']
)

print(f"Books shape: {df_books.shape}")
df_books.head()

In [ ]:
# Load users
df_users = pd.read_csv(
    os.path.join(path, "Users.csv"),
    usecols=['User-ID', 'Age']
)

print(f"Users shape: {df_users.shape}")
df_users.head()

In [ ]:
# Merge all datasets
df = df_ratings.merge(df_users, on='User-ID', how='left')
df = df.merge(df_books, on='ISBN', how='left')

print(f"Merged dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nRatings range: {df['Book-Rating'].min()} to {df['Book-Rating'].max()}")
print(f"Mean rating: {df['Book-Rating'].mean():.2f}")
print(f"\nMissing values:")
print(df.isna().sum())

## Step 3: DataVint Experiment Tracking

In [ ]:
# Start experiment
exp = dv.experiment("book_recommendations").__enter__()

# Log raw data (D0)
data_v0_id = exp.log_data(df, message="raw merged data")
print(f"✓ Logged D0: {data_v0_id}")
print(f"  Rows: {len(df)}, Columns: {len(df.columns)}")

## Step 4: Prepare Training Data

Create binary target: high_rating (>= 7)

In [ ]:
# Clean data
df_clean = df.dropna(subset=['Age', 'Year-Of-Publication'])

# Convert Year to numeric
df_clean['Year-Of-Publication'] = pd.to_numeric(
    df_clean['Year-Of-Publication'],
    errors='coerce'
)
df_clean = df_clean.dropna(subset=['Year-Of-Publication'])

# Create binary target
df_clean['high_rating'] = (df_clean['Book-Rating'] >= 7).astype(int)

print(f"Clean data shape: {df_clean.shape}")
print(f"High rating rate: {df_clean['high_rating'].mean():.2%}")

## Step 5: Sweep 1 - Test max_depth on Raw Data

In [ ]:
# Features: user age, book publication year
features = ['Age', 'Year-Of-Publication']
X = df_clean[features]
y = df_clean['high_rating']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Testing different max_depth values...\n")

results_sweep1 = []
for depth in [5, 10, 15]:
    # Train model
    model = XGBClassifier(
        max_depth=depth,
        n_estimators=100,
        learning_rate=0.03,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    model.fit(X_train, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    # Log run
    run_id = exp.log_run(
        data_commit_id=data_v0_id,
        metrics={"accuracy": round(accuracy, 3), "auc": round(auc, 3)},
        params={"max_depth": depth, "n_estimators": 100, "learning_rate": 0.03},
        message=f"max_depth={depth}",
        sweep_id=1,
        sweep_name="Max Depth (from D0)"
    )
    
    results_sweep1.append({
        'run_id': run_id,
        'max_depth': depth,
        'accuracy': accuracy,
        'auc': auc
    })
    
    print(f"{run_id}: depth={depth:2d} → accuracy={accuracy:.3f}, auc={auc:.3f}")

pd.DataFrame(results_sweep1)

## Step 6: Feature Engineering

Add new features:
- `book_age`: Current year - Year of Publication
- `age_group`: Binned user age (0-18, 18-30, 30-50, 50+)

In [ ]:
df_engineered = df_clean.copy()

# Add book age feature
current_year = 2024
df_engineered['book_age'] = current_year - df_engineered['Year-Of-Publication']

# Add age group feature
df_engineered['age_group'] = pd.cut(
    df_engineered['Age'],
    bins=[0, 18, 30, 50, 100],
    labels=[0, 1, 2, 3]
).cat.codes.astype(float)

print("New features added:")
print(df_engineered[['Age', 'age_group', 'Year-Of-Publication', 'book_age']].head(10))

# Log engineered data (D1)
data_v1_id = exp.log_data(df_engineered, message="added book_age and age_group")
print(f"\n✓ Logged D1: {data_v1_id}")
print(f"  Rows: {len(df_engineered)}, Columns: {len(df_engineered.columns)}")

## Step 7: Sweep 2 - Test learning_rate on Engineered Data

In [ ]:
# Features with new engineered features
features_v1 = ['Age', 'Year-Of-Publication', 'book_age', 'age_group']
X_v1 = df_engineered[features_v1]
y_v1 = df_engineered['high_rating']

X_train_v1, X_test_v1, y_train_v1, y_test_v1 = train_test_split(
    X_v1, y_v1, test_size=0.2, random_state=42
)

print("Testing different learning rates...\n")

results_sweep2 = []
best_auc = 0

for lr in [0.01, 0.03, 0.05]:
    # Train model
    model = XGBClassifier(
        max_depth=10,
        n_estimators=150,
        learning_rate=lr,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    model.fit(X_train_v1, y_train_v1)
    
    # Evaluate
    y_pred = model.predict(X_test_v1)
    y_prob = model.predict_proba(X_test_v1)[:, 1]
    accuracy = accuracy_score(y_test_v1, y_pred)
    auc = roc_auc_score(y_test_v1, y_prob)
    
    # Log run
    is_best = auc > best_auc
    run_id = exp.log_run(
        data_commit_id=data_v1_id,
        metrics={"accuracy": round(accuracy, 3), "auc": round(auc, 3)},
        params={"max_depth": 10, "n_estimators": 150, "learning_rate": lr},
        message=f"learning_rate={lr}",
        sweep_id=2,
        sweep_name="Learning Rate (from D1, depth=10)",
        best=is_best
    )
    
    results_sweep2.append({
        'run_id': run_id,
        'learning_rate': lr,
        'accuracy': accuracy,
        'auc': auc,
        'best': is_best
    })
    
    status = "⭐ BEST" if is_best else ""
    print(f"{run_id}: lr={lr:0.2f} → accuracy={accuracy:.3f}, auc={auc:.3f} {status}")
    
    if is_best:
        best_auc = auc

pd.DataFrame(results_sweep2)

## Step 8: View Lineage in Dashboard

Open in browser: http://localhost:5175/playground/book_recommendations

You'll see:
- **Top**: D0 (raw) and D1 (engineered) data commits
- **Bottom**: 6 model runs grouped by sweep
- **Lines**: Purple connections showing data → model lineage
- **Hover**: Metrics appear on hover

In [ ]:
# Close experiment
exp.__exit__(None, None, None)

print("✅ Experiment tracking complete!")
print(f"\n📊 View lineage: http://localhost:5175/playground/book_recommendations")
print(f"🔬 MLflow UI: http://localhost:5001")